# Ai_training-model-4 — O'zbekcha Chat AI

**Faqat o'zbek tilida** ishlovchi chat model trainingi.
- **Model:** ~30M params (embed=512, layers=8, heads=8, ff=2048)
- **Vocab:** 16k SentencePiece BPE
- **Vaqt:** 4-5 soat T4 / L4 GPU
- **Datasetlar:** behbudiy/alpaca-cleaned-uz, translation-instruction-uzbek, risqaliyevds/uzbek_instruct, wiki-uz

**Muhim:** `Runtime → Change runtime type → GPU (T4)` ni tanlang.

## 1. Drive mount va GPU tekshirish

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT  = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'
DRIVE_DATA = '/content/drive/MyDrive/Ai_chat_data_v4'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
print(f'[OUT]  {DRIVE_OUT}')
print(f'[DATA] {DRIVE_DATA}')
!nvidia-smi | head -10

## 2. Repoyni klon qilish

In [ ]:
%cd /content
!rm -rf Ai_training-model-4
!git clone https://github.com/Muhammadjonmurodullayev/Ai_training-model-4.git
%cd Ai_training-model-4
!pip install -q -r requirements.txt
print('[OK] repo + dependencies ready')

## 3. Datasetni tayyorlash (~10-15 daqiqa)

HuggingFace dan o'zbekcha datasetlarni yuklab oladi va ChatML JSONL ga aylantiradi.
Drive da bo'lsa qayta tayyorlamaydi.

In [ ]:
import os, shutil
TRAIN_F = f'{DRIVE_DATA}/chat_train.jsonl'
VAL_F   = f'{DRIVE_DATA}/chat_val.jsonl'

%cd /content/Ai_training-model-4
!mkdir -p data

if os.path.exists(TRAIN_F) and os.path.exists(VAL_F):
    print('[CACHE] Drive dan dataset olinadi')
    !cp "$TRAIN_F" data/
    !cp "$VAL_F"   data/
else:
    print('[NEW] Dataset tayyorlanmoqda (10-15 daqiqa)...')
    !python scripts/prepare_dataset.py \
      --output-dir data \
      --sources alpaca-uz,wiki-uz \
      --max-samples 200000 --val-ratio 0.01
    !cp data/chat_train.jsonl "$DRIVE_DATA/"
    !cp data/chat_val.jsonl   "$DRIVE_DATA/"

!ls -lh data/

## 4. Tokenizer training (~3 daqiqa)

In [ ]:
VOCAB_F = f'{DRIVE_OUT}/chat_vocab.model'

if os.path.exists(VOCAB_F):
    print('[CACHE] Drive dan vocab olinadi')
    !cp "$VOCAB_F" data/
    !cp "$DRIVE_OUT/chat_vocab.vocab" data/ 2>/dev/null || true
else:
    !python scripts/train_tokenizer.py \
      --train data/chat_train.jsonl \
      --val data/chat_val.jsonl \
      --output data/chat_vocab \
      --vocab-size 16000
    !cp data/chat_vocab.model "$DRIVE_OUT/"
    !cp data/chat_vocab.vocab "$DRIVE_OUT/"
    print('[OK] tokenizer Drive ga saqlandi')
!ls -lh data/chat_vocab.*

## 4b. Drive checkpoint MONITOR (alohida Colab tab da ishlatish)

**MUHIM:** Colab da bitta notebook ichida 2 ta cell parallel ishlamaydi. Shuning uchun:

1. **Yangi Colab tab oching:** https://colab.research.google.com → File → New notebook
2. Yangi notebookda **faqat 2 ta cell** kerak:
   - **Cell A:** Drive mount (pastdagi Cell 4b dan oldingisi)
     ```python
     from google.colab import drive
     drive.mount('/content/drive')
     ```
   - **Cell B:** monitor kodi (pastdagi)
3. **Cell A → Cell B** ni run qiling. Endi har 2 daqiqada chat_last.pt holati ko'rinib turadi.

> 💡 Monitor faqat Drive ni o'qiydi, training tezligiga ta'sir qilmaydi.

In [ ]:
# Monitor: Drive da chat_last.pt va chat_best.pt holatini har 120 sek da chiqaradi.
# Cell 5 ni ishga tushirgandan keyin shu cell ni alohida ishga tushiring (parallel).
# To'xtatish uchun: Runtime > Interrupt execution
import os, time, datetime
DRIVE_OUT = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'
os.makedirs(DRIVE_OUT, exist_ok=True)

print(f'[MONITOR] {DRIVE_OUT}\n')
while True:
    now = datetime.datetime.now().strftime('%H:%M:%S')
    rows = []
    for name in ('chat_last.pt', 'chat_best.pt', 'chat_vocab.model'):
        p = os.path.join(DRIVE_OUT, name)
        if os.path.exists(p):
            sz = os.path.getsize(p) / (1024*1024)
            mt = datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%H:%M:%S')
            rows.append(f'  {name:18s} {sz:7.1f} MB  modified {mt}')
        else:
            rows.append(f'  {name:18s} HALI YO\'Q')
    print(f'[{now}] Drive holati:')
    for r in rows: print(r)
    print()
    time.sleep(120)

## 5. Training (~4-5 soat)

**Muhim:** `--output-dir` to'g'ridan-to'g'ri **Drive** ga ishora qiladi.
Har `eval-every` stepda eng yaxshi `chat_best.pt` Drive ga yoziladi.
Sessiya uzilsa ham checkpoint xavfsiz.

In [ ]:
import os
DRIVE_OUT = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'
RESUME = f'{DRIVE_OUT}/chat_last.pt'

if os.path.exists(RESUME):
    print(f'[RESUME] {RESUME}')
    !python scripts/train_chat.py \
      --train data/chat_train.jsonl \
      --val   data/chat_val.jsonl \
      --tokenizer data/chat_vocab.model \
      --output-dir "$DRIVE_OUT" \
      --resume "$RESUME" \
      --embed-dim 512 --num-heads 8 --num-layers 8 \
      --ff-dim 2048 --max-seq-len 512 \
      --batch-size 16 --grad-accum 4 --epochs 6 --lr 3e-4 \
      --warmup-steps 500 --eval-every 1000 --save-every 200 --log-every 50
else:
    print('[ERR] chat_last.pt YO\'Q — Cell 5 dan boshlang')

## 6. Resume (sessiya uzilganda)

Agar Colab uzilib qolsa shu cell'ni 1-2 ni qayta ishga tushirib, keyin shu yerdan davom eting.

In [ ]:
import os, time
DRIVE_OUT = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'

assert os.path.exists('data/chat_train.jsonl'), 'Cell 3 ni run qiling!'
assert os.path.exists('data/chat_val.jsonl'),   'Cell 3 ni run qiling!'
assert os.path.exists('data/chat_vocab.model'), 'Cell 4 ni run qiling!'
assert os.path.isdir('/content/drive/MyDrive'), 'Drive mount qilinmagan! Cell 1 ni run qiling.'

print(f'[START] {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'[OUT]   {DRIVE_OUT}')
print('-' * 60)

# Tokenizer va vocab.model nomini chaqirish
!python scripts/train_chat.py \
  --train data/chat_train.jsonl \
  --val   data/chat_val.jsonl \
  --tokenizer data/chat_vocab.model \
  --output-dir "$DRIVE_OUT" \
  --embed-dim 512 \
  --num-heads 8 \
  --num-layers 8 \
  --ff-dim 2048 \
  --max-seq-len 512 \
  --batch-size 16 \
  --grad-accum 4 \
  --epochs 6 \
  --lr 3e-4 \
  --warmup-steps 500 \
  --eval-every 1000 \
  --save-every 200 \
  --log-every 50

## 7. Yakuniy ZIP yaratish

In [ ]:
import os, shutil
DRIVE_OUT = '/content/drive/MyDrive/Ai_chat_checkpoints_v4'

print('=== Drive dagi checkpointlar ===')
!ls -lh "$DRIVE_OUT/"

# Tokenizer ham bor-yo'qligini tekshir, agar yo'q bo'lsa local dan ko'chir
if not os.path.exists(f'{DRIVE_OUT}/chat_vocab.model') and os.path.exists('data/chat_vocab.model'):
    !cp data/chat_vocab.model "$DRIVE_OUT/"
    !cp data/chat_vocab.vocab "$DRIVE_OUT/" 2>/dev/null || true
    print('[FIX] vocab Drive ga ko\'chirildi')

# ZIP yaratish
ZIP = '/content/checkpoints_v4.zip'
shutil.make_archive(ZIP[:-4], 'zip', DRIVE_OUT)
sz = os.path.getsize(ZIP) / (1024*1024)
print(f'\n[ZIP] {ZIP}  ({sz:.1f} MB)')

# Drive ga ham nusxa
shutil.copy(ZIP, '/content/drive/MyDrive/checkpoints_v4.zip')
print('[OK] Drive: /content/drive/MyDrive/checkpoints_v4.zip — yuklab oling!')